# Langfuse Demo 2 — RAG Observability

**Dataset:** `ecommerce_support_requests.csv`  
**Runtime:** Python 3.11.9  
**Purpose:** A simple, instructor-led Langfuse demonstration.

## Architecture

```text
CSV rows → text documents → TF-IDF index → retrieve top support records → prompt → LLM answer
                                  ↓                                  ↓
                           retrieval span                 generation trace
```

The CSV acts as a tiny knowledge base. Langfuse records the retrieval step and the traced LLM generation. This demo uses local TF-IDF retrieval so no embedding API is required.

## Step 1 — Install packages

In [ ]:
%pip install -q -U langfuse openai pandas scikit-learn python-dotenv

## Step 2 — Load the dataset

In [ ]:
import os, re, json
import pandas as pd

# Keep ecommerce_support_requests.csv in the same folder as this notebook.
CSV_FILE = "ecommerce_support_requests.csv"
df = pd.read_csv(CSV_FILE)
print(f"Loaded {len(df)} rows from {CSV_FILE}")
display(df.head(3))

## Step 3 — Load credentials from `.env`

Place a `.env` file beside the notebook:

```env
OPENAI_API_KEY=your-openai-key
LANGFUSE_PUBLIC_KEY=your-langfuse-public-key
LANGFUSE_SECRET_KEY=your-langfuse-secret-key
LANGFUSE_BASE_URL=https://cloud.langfuse.com
```

In [ ]:
from dotenv import load_dotenv

# Loads variables from a .env file in the notebook's current directory.
load_dotenv()

required_keys = ["OPENAI_API_KEY", "LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY"]
missing_keys = [key for key in required_keys if not os.getenv(key)]
if missing_keys:
    raise ValueError(f"Missing variables in .env: {', '.join(missing_keys)}")

# Keep this in .env when using another Langfuse region or a self-hosted instance.
os.environ.setdefault("LANGFUSE_BASE_URL", "https://cloud.langfuse.com")

from langfuse import get_client
langfuse = get_client()
print("Langfuse authentication:", langfuse.auth_check())

## Step 4 — Prepare masked documents and build a local retriever

In [ ]:
def mask_pii(value):
    """Mask likely email addresses and 10-digit phone numbers before tracing."""
    text = str(value)
    text = re.sub(r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}", "<EMAIL>", text)
    text = re.sub(r"(?<!\d)\d{10}(?!\d)", "<PHONE>", text)
    return text


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def row_to_document(row):
    return (
        f"Request {row.request_id}; order {row.order_id}; category {row.product_category}; "
        f"status {row.order_status}; tier {row.customer_tier}; issue {row.issue_type}; "
        f"message {mask_pii(row.customer_message)}"
    )

documents = [row_to_document(row) for row in df.itertuples(index=False)]
vectorizer = TfidfVectorizer(stop_words="english")
document_matrix = vectorizer.fit_transform(documents)
print(documents[0])

## Step 5 — Trace the retrieval operation

`as_type="retriever"` makes the retrieval step visually distinct inside the trace.

In [ ]:
from langfuse import observe, propagate_attributes

@observe(name="retrieve-support-records", as_type="retriever")
def retrieve(query, top_k=3):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, document_matrix)[0]
    best = scores.argsort()[::-1][:top_k]
    return [{"text": documents[i], "score": round(float(scores[i]), 4)} for i in best]

display(retrieve("What is the refund status for ORD-5003?"))

## Step 6 — Build and run the traced RAG pipeline

In [ ]:
from langfuse.openai import OpenAI
client = OpenAI()

@observe(name="ecommerce-rag-pipeline")
def answer_with_rag(question, session_id="rag-demo-session"):
    safe_question = mask_pii(question)
    with propagate_attributes(
        trace_name="ecommerce-rag",
        session_id=session_id,
        tags=["training", "rag", "ecommerce"],
        metadata={"top_k": "3", "retriever": "tfidf"},
    ):
        matches = retrieve(safe_question, top_k=3)
        context = "\n".join(item["text"] for item in matches)
        completion = client.chat.completions.create(
            name="rag-answer-generation",
            model="gpt-4.1-mini",
            temperature=0,
            messages=[
                {"role": "system", "content": "Answer only from CONTEXT. If unsupported, say you do not know."},
                {"role": "user", "content": f"QUESTION:\n{safe_question}\n\nCONTEXT:\n{context}"},
            ],
        )
        return {"answer": completion.choices[0].message.content, "contexts": matches}

result = answer_with_rag("What is the refund status for order ORD-5003?")
print(result["answer"])
display(pd.DataFrame(result["contexts"]))
langfuse.flush()

## Step 7 — Simple retrieval evaluation

For a known order query, we check whether the expected order ID appeared in the retrieved context. This is a transparent code-based metric for classroom use.

In [ ]:
test_cases = [
    {"question": "What is happening with ORD-5003?", "expected_order": "ORD-5003"},
    {"question": "Give me the status of ORD-5001", "expected_order": "ORD-5001"},
    {"question": "What happened to ORD-5002?", "expected_order": "ORD-5002"},
]

evaluation_rows = []
for case in test_cases:
    hits = retrieve(case["question"], top_k=3)
    retrieved_text = " ".join(x["text"] for x in hits)
    evaluation_rows.append({
        **case,
        "retrieval_hit": int(case["expected_order"] in retrieved_text),
        "top_score": hits[0]["score"],
    })

evaluation_df = pd.DataFrame(evaluation_rows)
display(evaluation_df)
print("Hit rate:", evaluation_df["retrieval_hit"].mean())
langfuse.flush()

## Step 8 — Langfuse monitoring walkthrough

1. Open the `ecommerce-rag` trace.
2. Expand `retrieve-support-records` and inspect the top three records and similarity scores.
3. Verify **retrieval relevance**: the expected order or topic should appear in the retrieved records.
4. Open `rag-answer-generation` and inspect the question, supplied context and final answer.
5. Check **grounding**: every factual statement in the answer should be supported by retrieved context.
6. Compare **retrieval latency** with **LLM generation latency** to locate the bottleneck.
7. Check token usage and cost; large retrieved contexts normally increase both.
8. Inspect `top_k`, `retriever`, session ID, tags and other metadata.
9. Compare the code-calculated retrieval hit rate with trace-level evaluation results.
10. Add or configure scores for context precision, context recall, faithfulness and answer relevance.
11. Check errors separately for retrieval failure, empty context and generation failure.
12. Create dashboard charts for RAG latency, token use, cost, hit rate and quality scores.

### Recommended RAG alerts

- No context returned by the retriever.
- Expected record absent from the top results.
- Low top similarity score.
- Low faithfulness or answer relevance.
- Excessive context size, latency or cost.
- PII appearing in retrieved context or model output.